# División de Dataset de Imágenes
### Train 70% | Val 15% | Test 15%

Diseñado para ~20,000 imágenes sin sobrecargar el procesamiento.
**Solo mueve/copia archivos** — no carga imágenes en memoria.

In [1]:
# ── Instalación (solo si hace falta) ──────────────────────────────────────────
# tqdm ya viene con Anaconda/pip; si no:
# !pip install tqdm

In [2]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN  ← edita solo esta sección
# ─────────────────────────────────────────────────────────────────────────────
SOURCE_DIR   = Path("data")          # carpeta con tus 20k imágenes
OUTPUT_DIR   = Path("dataset")       # carpeta de salida (se crea automáticamente)

TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15                  # debe sumar 1.0 con las anteriores

SEED         = 42                    # para reproducibilidad
COPY_FILES   = False                  # True = copia  |  False = mueve (más rápido, libera espacio)
EXTENSIONS   = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
CHUNK_SIZE   = 500                   # archivos procesados por lote (controla uso de memoria)
# ─────────────────────────────────────────────────────────────────────────────

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, \
    "Las proporciones deben sumar exactamente 1.0"

print("✅ Configuración cargada")
print(f"   Origen  : {SOURCE_DIR.resolve()}")
print(f"   Destino : {OUTPUT_DIR.resolve()}")
print(f"   Modo    : {'COPIA' if COPY_FILES else 'MOVER'}")

✅ Configuración cargada
   Origen  : /home/daniela/Documentos/ia_code/workshop2/regresion/data
   Destino : /home/daniela/Documentos/ia_code/workshop2/regresion/dataset
   Modo    : MOVER


In [3]:
# ── 1. Listar imágenes (sin cargarlas en memoria) ─────────────────────────────
print("🔍 Escaneando imágenes...")

all_images = [
    p for p in SOURCE_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in EXTENSIONS
]

total = len(all_images)
print(f"   Total encontradas: {total:,} imágenes")

if total == 0:
    raise FileNotFoundError(
        f"No se encontraron imágenes en '{SOURCE_DIR}'. "
        f"Verifica la ruta y las extensiones: {EXTENSIONS}"
    )

🔍 Escaneando imágenes...
   Total encontradas: 24,106 imágenes


In [4]:
# ── 2. Mezcla aleatoria y división ────────────────────────────────────────────
random.seed(SEED)
random.shuffle(all_images)

n_train = int(total * TRAIN_RATIO)
n_val   = int(total * VAL_RATIO)
n_test  = total - n_train - n_val   # el resto va a test (evita redondeo)

splits = {
    "train" : all_images[:n_train],
    "val"   : all_images[n_train : n_train + n_val],
    "test"  : all_images[n_train + n_val :],
}

print("📊 División calculada:")
for name, files in splits.items():
    pct = len(files) / total * 100
    print(f"   {name:6s}: {len(files):>6,} imágenes  ({pct:.1f}%)")

📊 División calculada:
   train : 16,874 imágenes  (70.0%)
   val   :  3,615 imágenes  (15.0%)
   test  :  3,617 imágenes  (15.0%)


In [5]:
# ── 3. Crear carpetas destino ─────────────────────────────────────────────────
for split_name in splits:
    dest = OUTPUT_DIR / split_name
    dest.mkdir(parents=True, exist_ok=True)

print(f"📁 Carpetas creadas en: {OUTPUT_DIR.resolve()}")

📁 Carpetas creadas en: /home/daniela/Documentos/ia_code/workshop2/regresion/dataset


In [6]:
# ── 4. Copiar/mover en lotes (CHUNK_SIZE) para no saturar el sistema ──────────
def process_in_chunks(file_list, dest_folder, operation="copy", chunk_size=500):
    """Procesa archivos en lotes pequeños para controlar el I/O."""
    total_files = len(file_list)
    errors = []
    
    fn = shutil.copy2 if operation == "copy" else shutil.move

    with tqdm(total=total_files, unit="img", dynamic_ncols=True) as pbar:
        for start in range(0, total_files, chunk_size):
            chunk = file_list[start : start + chunk_size]
            for src in chunk:
                dst = dest_folder / src.name
                # Evitar colisiones de nombre
                if dst.exists():
                    dst = dest_folder / f"{src.stem}_{src.stat().st_ino}{src.suffix}"
                try:
                    fn(src, dst)
                except Exception as e:
                    errors.append((src, str(e)))
                pbar.update(1)
    
    return errors


operation = "copy" if COPY_FILES else "move"
all_errors = {}

for split_name, file_list in splits.items():
    print(f"\n{'📋' if COPY_FILES else '🚚'} Procesando [{split_name}] — {len(file_list):,} archivos...")
    dest_folder = OUTPUT_DIR / split_name
    errs = process_in_chunks(file_list, dest_folder, operation, CHUNK_SIZE)
    if errs:
        all_errors[split_name] = errs
        print(f"   ⚠️  {len(errs)} error(es) en '{split_name}'")

print("\n✅ Proceso completado.")


🚚 Procesando [train] — 16,874 archivos...


  0%|          | 0/16874 [00:00<?, ?img/s]

100%|██████████| 16874/16874 [00:01<00:00, 15938.00img/s]



🚚 Procesando [val] — 3,615 archivos...


100%|██████████| 3615/3615 [00:00<00:00, 12684.55img/s]



🚚 Procesando [test] — 3,617 archivos...


100%|██████████| 3617/3617 [00:00<00:00, 17460.84img/s]


✅ Proceso completado.


In [7]:
# ── 5. Verificación final ─────────────────────────────────────────────────────
print("\n🔎 Verificación de archivos en destino:")
grand_total = 0
for split_name in splits:
    dest = OUTPUT_DIR / split_name
    count = sum(1 for f in dest.iterdir() if f.suffix.lower() in EXTENSIONS)
    expected = len(splits[split_name])
    status = "✅" if count == expected else "⚠️ "
    print(f"   {status} {split_name:6s}: {count:>6,} / {expected:,} esperadas")
    grand_total += count

print(f"\n   Total en dataset/: {grand_total:,}")

if all_errors:
    print("\n⚠️  Errores detectados:")
    for split_name, errs in all_errors.items():
        print(f"   [{split_name}]")
        for path, msg in errs[:5]:   # muestra hasta 5
            print(f"     • {path.name}: {msg}")
else:
    print("\n🎉 Sin errores. Dataset listo para entrenar.")


🔎 Verificación de archivos en destino:
   ✅ train : 16,874 / 16,874 esperadas
   ✅ val   :  3,615 / 3,615 esperadas
   ✅ test  :  3,617 / 3,617 esperadas

   Total en dataset/: 24,106

🎉 Sin errores. Dataset listo para entrenar.


In [8]:
# ── 6. (Opcional) Guardar lista de archivos por split en CSV ──────────────────
import csv

log_path = OUTPUT_DIR / "split_log.csv"
with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "split"])
    for split_name, file_list in splits.items():
        for p in file_list:
            writer.writerow([p.name, split_name])

print(f"📄 Log guardado en: {log_path}")
print("   Útil para reproducir la misma división o rastrear qué imagen va a cada split.")

📄 Log guardado en: dataset/split_log.csv
   Útil para reproducir la misma división o rastrear qué imagen va a cada split.
